# NLP — Language Modeling (N-grams)

## الترتيب / Flow
1. استيراد المكتبات - Import libraries
2. تجهيز corpus - Prepare corpus
3. Unigrams & bigrams - Build n-grams
4. احتمالات N-gram - N-gram probabilities
5. Perplexity - Evaluate language model
6. توليد نص بسيط - Simple text generation

In [ ]:
# Step 1) استيراد المكتبات / Import libraries
# pip install nltk -q
import math
import nltk
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from collections import Counter

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
# Step 2) تجهيز corpus / Prepare corpus
corpus = [
    "natural language processing enables computers to understand text",
    "language models predict the next word in a sequence",
    "n gram models are simple but useful language models",
]

tokens = []
for sentence in corpus:
    tokens.extend(['<s>'] + word_tokenize(sentence.lower()) + ['</s>'])

print(f"Total tokens: {len(tokens)}")
print(tokens[:15])

In [ ]:
# Step 3) Unigrams & bigrams / بناء N-grams
unigrams = Counter(tokens)
bigrams = Counter(ngrams(tokens, 2))
trigrams = Counter(ngrams(tokens, 3))

print("Top unigrams:", unigrams.most_common(8))
print("Top bigrams:", bigrams.most_common(8))

In [ ]:
# Step 4) احتمالات N-gram / N-gram probabilities (with smoothing)
V = len(unigrams)

def bigram_prob(w1, w2, alpha=1.0):
  return (bigrams[(w1, w2)] + alpha) / (unigrams[w1] + alpha * V)

examples = [('language', 'models'), ('language', 'processing'), ('the', 'next')]
for w1, w2 in examples:
    print(f"P({w2}|{w1}) = {bigram_prob(w1, w2):.4f}")

In [ ]:
# Step 5) Perplexity / قياس جودة النموذج
def sentence_log_prob(sentence_tokens, alpha=1.0):
    seq = ['<s>'] + sentence_tokens + ['</s>']
    log_p = 0.0
    for w1, w2 in zip(seq[:-1], seq[1:]):
        log_p += math.log(bigram_prob(w1, w2, alpha=alpha))
    return log_p

def perplexity(sentence_tokens, alpha=1.0):
    n = len(sentence_tokens) + 1
    return math.exp(-sentence_log_prob(sentence_tokens, alpha) / n)

test_sentences = [
    ['language', 'models', 'predict', 'the', 'next', 'word'],
    ['natural', 'language', 'processing', 'enables', 'computers'],
]

for sent in test_sentences:
    print(f"{' '.join(sent):<45} perplexity = {perplexity(sent):.2f}")

In [ ]:
# Step 6) توليد نص بسيط / Simple bigram generation
import random

def generate(max_len=8, start='<s>'):
    word = start
    output = []
    for _ in range(max_len):
        candidates = [w2 for (w1, w2) in bigrams if w1 == word]
        if not candidates:
            break
        word = random.choice(candidates)
        if word == '</s>':
            break
        output.append(word)
    return ' '.join(output)

for i in range(3):
    print(generate())